In [15]:
from pathlib import Path
import re
import json
import hashlib
from collections import Counter
from transformers import AutoTokenizer
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [16]:
DATA_ROOT = Path(
    r"C:\Users\kgathola.puka\OneDrive - MSC\Documents\GitHub\RCP(test)\SPEED CHATBOT PROJECT\DATA\Cleaned_Generative"
)
OUTPUT_DIR = Path(
    r"C:\Users\kgathola.puka\OneDrive - MSC\Documents\GitHub\RCP(test)\SPEED CHATBOT PROJECT\DATA\unified_semantic_chunks"
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SKIP_FOLDERS = {"unified_semantic_chunks", "vector_store", "Cleaned_Generative","Extracted"}

print(f"📂 Input:  {DATA_ROOT}")
print(f"📂 Output: {OUTPUT_DIR}")

📂 Input:  C:\Users\kgathola.puka\OneDrive - MSC\Documents\GitHub\RCP(test)\SPEED CHATBOT PROJECT\DATA\Cleaned_Generative
📂 Output: C:\Users\kgathola.puka\OneDrive - MSC\Documents\GitHub\RCP(test)\SPEED CHATBOT PROJECT\DATA\unified_semantic_chunks


In [17]:
MODEL_NAME       = "sentence-transformers/all-MiniLM-L6-v2"
tokenizer        = AutoTokenizer.from_pretrained(MODEL_NAME)
MAX_MODEL_TOKENS = 512
SAFE_CHUNK_SIZE  = 450
CHUNK_OVERLAP    = 50
MIN_TOKENS       = 10   # chunks below this are noise

def count_tokens(text: str) -> int:
    return len(tokenizer.encode(text, add_special_tokens=False))

def normalize_text(text: str) -> str:
    text = re.sub(r"\r\n", "\n", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

def make_chunk(text, chunk_type, source, category, chunk_id, section_id,
               structured_data=None, extra_meta=None):
    """Unified chunk factory — guarantees consistent shape across all chunk types."""
    meta = {
        "category"    : category,
        "source"      : source,
        "chunk_type"  : chunk_type,
        **( extra_meta or {})
    }
    return {
        "text"           : text,
        "metadata"       : meta,
        "structured_data": structured_data,
        "section_id"     : section_id,
        "chunk_id"       : chunk_id,
        "tokens"         : count_tokens(text)
    }

In [18]:
def process_normal_text(file_path: Path, category: str) -> list:
    """
    Reads a cleaned TXT file, respects [META], [PAGE N], [TABLE] markers.
    Prose and table blocks are chunked separately and tagged accordingly.
    """
    raw = file_path.read_text(encoding="utf-8")

    # --- Extract META block ---
    file_meta = {}
    meta_match = re.search(r"\[META\](.*?)\[/META\]", raw, re.DOTALL)
    if meta_match:
        for line in meta_match.group(1).strip().split("\n"):
            if ":" in line:
                k, v = line.split(":", 1)
                file_meta[k.strip()] = v.strip()
        raw = raw[meta_match.end():]

    # --- Split by [PAGE N] markers ---
    page_blocks = re.split(r"\[PAGE (\d+)\]", raw)

    chunks   = []
    chunk_id = 0

    splitter = RecursiveCharacterTextSplitter(
        chunk_size    = SAFE_CHUNK_SIZE,
        chunk_overlap = CHUNK_OVERLAP,
        length_function = count_tokens,
        separators    = ["\n\n", "\n", ". ", " ", ""]
    )

    it = iter(page_blocks)
    next(it)  # skip any content before first PAGE marker

    for page_num, page_text in zip(it, it):
        page_text = normalize_text(page_text)
        if not page_text:
            continue

        # --- Extract table blocks from this page ---
        table_blocks = re.findall(r"\[TABLE\](.*?)\[/TABLE\]", page_text, re.DOTALL)
        prose = re.sub(r"\[TABLE\].*?\[/TABLE\]", "", page_text, flags=re.DOTALL)
        prose = normalize_text(prose)

        extra = {
            "page_number"  : int(page_num),
            "is_table_schema": False,
            "source_file"  : file_path.name
        }

        # --- Chunk prose ---
        if prose:
            for section in splitter.split_text(prose):
                # Safety truncation
                if count_tokens(section) > MAX_MODEL_TOKENS:
                    section = tokenizer.decode(
                        tokenizer.encode(section, truncation=True, max_length=SAFE_CHUNK_SIZE),
                        skip_special_tokens=True
                    )
                chunks.append(make_chunk(
                    text         = section,
                    chunk_type   = "text_prose",
                    source       = file_path.name,
                    category     = category,
                    chunk_id     = chunk_id,
                    section_id   = int(page_num),
                    extra_meta   = {**extra, "content_type": "prose"}
                ))
                chunk_id += 1

        # --- Each table block = one chunk (keep intact) ---
        for table_text in table_blocks:
            table_text = table_text.strip()
            if not table_text:
                continue
            chunks.append(make_chunk(
                text         = table_text,
                chunk_type   = "text_table",
                source       = file_path.name,
                category     = category,
                chunk_id     = chunk_id,
                section_id   = int(page_num),
                extra_meta   = {**extra, "content_type": "table"}
            ))
            chunk_id += 1

    return chunks

In [19]:
def process_table_schema_json(file_path: Path, category: str) -> list:
    with open(file_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    if not isinstance(data, dict):
        return []

    table_name  = data.get("table_name", file_path.stem)
    description = data.get("description", "N/A")
    primary_key = data.get("primary_key", "N/A")
    columns     = data.get("columns", [])

    pk_cols    = [c for c in columns if c.get("is_primary_key")]
    fk_cols    = [c for c in columns if c.get("is_foreign_key")]
    alpha_cols = [c for c in columns if "ALPHA" in c["name"] or "NUM" in c["name"]
                                     or "TOP"   in c["name"] or "DATE" in c["name"]
                                     or "HEURE" in c["name"]]
    core_cols  = [c for c in columns if c not in pk_cols
                                     and c not in fk_cols
                                     and c not in alpha_cols]

    def fmt_col(c):
        return f"{c['name']} ({c.get('type_sql_server','?')}): {c.get('description','')}"

    def fmt_fk(c):
        return (f"{c['name']} → {c.get('references_table','?')}.{c.get('references_column','?')}"
                f" ({c.get('description','')})")

    chunks   = []
    chunk_id = 0

    base_meta = {
        "is_table_schema": True,
        "table_name"     : table_name,
        "source_file"    : file_path.name,
        "related_tables" : list({c["references_table"] for c in fk_cols
                                  if c.get("references_table")})
    }

    # --- Chunk 1: Table overview + FK relationships ---
    fk_lines = [fmt_fk(c) for c in fk_cols] if fk_cols else ["None"]

    overview_lines = [
        f"TABLE: {table_name}",
        f"DESCRIPTION: {description}",
        f"PRIMARY KEY: {primary_key}",
        f"CATEGORY: {category}",
        "",
        "FOREIGN KEY RELATIONSHIPS:",
    ]
    overview_lines.extend(fk_lines)

    chunks.append(make_chunk(
        text            = "\n".join(overview_lines),
        chunk_type      = "schema_overview",
        source          = file_path.name,
        category        = category,
        chunk_id        = chunk_id,
        section_id      = 0,
        structured_data = data,
        extra_meta      = base_meta
    ))
    chunk_id += 1

    # --- Chunk 2: Core business columns ---
    if core_cols:
        core_lines = [f"TABLE: {table_name} — CORE COLUMNS"]
        core_lines.extend([fmt_col(c) for c in core_cols])

        chunks.append(make_chunk(
            text       = "\n".join(core_lines),
            chunk_type = "schema_core_columns",
            source     = file_path.name,
            category   = category,
            chunk_id   = chunk_id,
            section_id = 1,
            extra_meta = base_meta
        ))
        chunk_id += 1

    # --- Chunk 3: Additional/generic columns ---
    if alpha_cols:
        alpha_lines = [f"TABLE: {table_name} — ADDITIONAL/GENERIC COLUMNS"]
        alpha_lines.extend([fmt_col(c) for c in alpha_cols])

        chunks.append(make_chunk(
            text       = "\n".join(alpha_lines),
            chunk_type = "schema_extra_columns",
            source     = file_path.name,
            category   = category,
            chunk_id   = chunk_id,
            section_id = 2,
            extra_meta = base_meta
        ))
        chunk_id += 1

    return chunks

In [20]:
def process_wms_reference_json(file_path: Path, category: str) -> list:
    """
    Processes a WMS reference JSON (e.g. Check Loading Details.json).
    Each procedure gets its own chunk. Join logic and safety rules
    are chunked separately so they can be retrieved independently.
    """
    with open(file_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    if not isinstance(data, dict):
        return []

    doc_name    = data.get("document_name", file_path.stem)
    version     = data.get("version", "N/A")
    core_tables = data.get("core_tables", {})
    join_logic  = data.get("join_logic", {})
    procedures  = data.get("procedures", [])
    safety_rules= data.get("safety_rules", [])

    chunks   = []
    chunk_id = 0

    base_meta = {
        "is_table_schema" : False,
        "document_type"   : "WMS Reference",
        "document_name"   : doc_name,
        "source_file"     : file_path.name,
        "related_tables"  : list(core_tables.keys())
    }

    # --- Chunk 1: Document overview + core tables ---
    overview_lines = [
        f"DOCUMENT: {doc_name}",
        f"VERSION: {version}",
        f"CATEGORY: {category}",
        f"DOCUMENT TYPE: WMS Reference",
        "",
        "CORE TABLES INVOLVED:",
        *[f"  {tbl}: {desc}" for tbl, desc in core_tables.items()]
    ]
    chunks.append(make_chunk(
        text         = "\n".join(overview_lines),
        chunk_type   = "wms_overview",
        source       = file_path.name,
        category     = category,
        chunk_id     = chunk_id,
        section_id   = 0,
        structured_data = {"document_name": doc_name, "core_tables": core_tables},
        extra_meta   = base_meta
    ))
    chunk_id += 1

    # --- Chunk 2: Join logic (inbound + outbound) ---
    if join_logic:
        join_lines = [f"DOCUMENT: {doc_name} — JOIN LOGIC", ""]
        for direction, info in join_logic.items():
            join_lines += [
                f"  {direction.upper()}:",
                f"    Primary Keys : {', '.join(info.get('primary_keys', []))}",
                f"    Description  : {info.get('description', 'N/A')}",
                f"    Example      : {info.get('example_pattern', 'N/A')}",
                ""
            ]
        chunks.append(make_chunk(
            text         = "\n".join(join_lines),
            chunk_type   = "wms_join_logic",
            source       = file_path.name,
            category     = category,
            chunk_id     = chunk_id,
            section_id   = 1,
            structured_data = join_logic,
            extra_meta   = base_meta
        ))
        chunk_id += 1

    # --- Chunk 3+: One chunk per procedure (with SQL) ---
    for proc in procedures:
        proc_name  = proc.get("procedure_name", "Unknown")
        sql        = proc.get("query", {}).get("sql", "N/A")
        proc_lines = [
            f"PROCEDURE: {proc_name}",
            f"DOCUMENT: {doc_name}",
            f"CATEGORY: {proc.get('category', 'N/A')}",
            f"ACCESS LEVEL: {proc.get('access_level', 'N/A')}",
            f"BUSINESS LOGIC: {proc.get('business_logic', 'N/A')}",
            "",
            "SQL:",
            sql
        ]
        chunks.append(make_chunk(
            text         = "\n".join(proc_lines),
            chunk_type   = "wms_procedure",
            source       = file_path.name,
            category     = category,
            chunk_id     = chunk_id,
            section_id   = 2,
            structured_data = proc,
            extra_meta   = {**base_meta, "procedure_name": proc_name}
        ))
        chunk_id += 1

    # --- Final chunk: Safety rules ---
    if safety_rules:
        rules_lines = [
            f"DOCUMENT: {doc_name} — SAFETY RULES",
            "",
            *[f"  {i+1}. {rule}" for i, rule in enumerate(safety_rules)]
        ]
        chunks.append(make_chunk(
            text         = "\n".join(rules_lines),
            chunk_type   = "wms_safety_rules",
            source       = file_path.name,
            category     = category,
            chunk_id     = chunk_id,
            section_id   = 3,
            structured_data = safety_rules,
            extra_meta   = base_meta
        ))
        chunk_id += 1

    return chunks

In [21]:
def dedup_chunks(chunks: list) -> list:
    seen, unique = set(), []
    for chunk in chunks:
        h = hashlib.md5(chunk["text"].strip().lower().encode()).hexdigest()
        if h not in seen:
            seen.add(h)
            unique.append(chunk)
    print(f"🧹 Dedup: removed {len(chunks) - len(unique)} duplicates "
          f"({len(unique)} remaining)")
    return unique

def filter_chunks(chunks: list) -> list:
    filtered = [c for c in chunks if c["tokens"] >= MIN_TOKENS]
    print(f"🔍 Quality filter: removed {len(chunks) - len(filtered)} "
          f"low-quality chunks")
    return filtered

In [22]:
def print_chunk_stats(chunks: list):
    types      = Counter(c["metadata"].get("chunk_type", "unknown") for c in chunks)
    categories = Counter(c["metadata"].get("category",   "unknown") for c in chunks)
    tokens     = [c["tokens"] for c in chunks]

    print(f"\n{'='*55}")
    print(f"📊 Chunk Statistics")
    print(f"{'='*55}")
    print(f"  Total chunks       : {len(chunks)}")
    print(f"  Avg tokens/chunk   : {sum(tokens)/len(tokens):.1f}")
    print(f"  Max tokens/chunk   : {max(tokens)}")
    print(f"  Min tokens/chunk   : {min(tokens)}")
    print(f"\n  By chunk_type:")
    for k, v in types.most_common():
        print(f"    {k:<30} : {v}")
    print(f"\n  By category:")
    for k, v in categories.most_common():
        print(f"    {k:<30} : {v}")
    print(f"{'='*55}")

In [23]:
def detect_json_type(data: dict) -> str:
    """
    Detects whether a JSON file is a WMS reference doc or a table schema.
    Uses key presence rather than brittle filename matching.
    """
    if isinstance(data, dict):
        if "core_tables" in data or "procedures" in data or "join_logic" in data:
            return "wms_reference"
        if "table_name" in data and "columns" in data:
            return "table_schema"
    return "unknown"

In [24]:
all_chunks = []

for category_dir in DATA_ROOT.iterdir():
    if not category_dir.is_dir():
        continue
    if category_dir.name in SKIP_FOLDERS:
        continue

    category = category_dir.name
    print(f"\n📁 Category: {category}")

    # --- TXT files ---
    txt_files = list(category_dir.glob("*.txt"))
    if not txt_files:
        print("   (No TXT files)")
    for file in txt_files:
        print(f"  📝 TXT: {file.name}")
        all_chunks.extend(process_normal_text(file, category))

    # --- JSON files ---
    json_files = [f for f in category_dir.glob("*.json")
                  if f.name != "unified_chunks.json"]
    if not json_files:
        print("   (No JSON files)")
    for file in json_files:
        with open(file, "r", encoding="utf-8") as f:
            data = json.load(f)

        json_type = detect_json_type(data)
        print(f"  ⚡ JSON: {file.name}  → detected as: {json_type}")

        if json_type == "wms_reference":
            all_chunks.extend(process_wms_reference_json(file, category))
        elif json_type == "table_schema":
            all_chunks.extend(process_table_schema_json(file, category))
        else:
            print(f"  ⚠️  Unknown JSON type — skipping: {file.name}")

Token indices sequence length is longer than the specified maximum sequence length for this model (2067 > 512). Running this sequence through the model will result in indexing errors



📁 Category: Database Tables
   (No TXT files)
  ⚡ JSON: ACT_PAR.JSON  → detected as: table_schema
  ⚡ JSON: ART_PAR.JSON  → detected as: table_schema
  ⚡ JSON: CHG_DAT.json  → detected as: table_schema
  ⚡ JSON: CHL_DAT .json  → detected as: table_schema
  ⚡ JSON: CHL_DAT.json  → detected as: table_schema
  ⚡ JSON: MIE_DAT.json  → detected as: table_schema
  ⚡ JSON: MIL_DAT.json  → detected as: table_schema
  ⚡ JSON: MVT_DAT.json  → detected as: table_schema
  ⚡ JSON: OPE_DAT.json  → detected as: table_schema
  ⚡ JSON: OPL_DAT.json  → detected as: table_schema
  ⚡ JSON: QUA_PAR.json  → detected as: table_schema
  ⚡ JSON: REA_DAT.json  → detected as: table_schema
  ⚡ JSON: REE_DAT.json  → detected as: table_schema
  ⚡ JSON: REL_DAT.json  → detected as: table_schema
  ⚡ JSON: SEX_DAT.json  → detected as: table_schema
  ⚡ JSON: stk_dat.json  → detected as: table_schema
  ⚡ JSON: TIE_PAR.JSON  → detected as: table_schema
  ⚡ JSON: ZEM_DAT.json  → detected as: table_schema

📁 Category: GEN

In [ ]:
# Dedup and filter
all_chunks = dedup_chunks(all_chunks)
all_chunks = filter_chunks(all_chunks)

# Reassign chunk IDs sequentially after dedup
for i, chunk in enumerate(all_chunks):
    chunk["chunk_id"] = i

# Stats
print_chunk_stats(all_chunks)

# Save
output_file = OUTPUT_DIR / "unified_chunks.json"
with open(output_file, "w", encoding="utf-8") as f:
    json.dump(all_chunks, f, indent=2, ensure_ascii=False)

print(f"\n💾 Saved {len(all_chunks)} chunks → {output_file}")

🧹 Dedup: removed 30 duplicates (631 remaining)
🔍 Quality filter: removed 4 low-quality chunks

📊 Chunk Statistics
  Total chunks       : 627
  Avg tokens/chunk   : 186.5
  Max tokens/chunk   : 2067
  Min tokens/chunk   : 11

  By chunk_type:
    text_prose                     : 470
    wms_procedure                  : 28
    text_table                     : 25
    schema_overview                : 18
    wms_overview                   : 18
    wms_join_logic                 : 18
    wms_safety_rules               : 18
    schema_core_columns            : 17
    schema_extra_columns           : 15

  By category:
    Speed Support Document         : 184
    LOREAL                         : 154
    GENERAL                        : 81
    Database Tables                : 50
    ORDER PREPARATION              : 47
    RECEIVING GOODS                : 44
    LOADING                        : 20
    REPLENISHMENT OF THE PICKING LOCATION : 10
    Support Ticket Docs            : 8
    MANUFACTU

: 